In [2]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters chromadb pypdf openai -q

In [3]:
import os
import openai

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from chromadb.config import Settings

/tmp/ipykernel_5758/2720623074.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
import os
from google.colab import files

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    UnstructuredWordDocumentLoader
)

# Upload Travel & Tourism file
uploaded = files.upload()

file_path = list(uploaded.keys())[0]

print("Uploaded Travel & Tourism File:", file_path)

# Detect file extension
ext = os.path.splitext(file_path)[1].lower()

# Select appropriate loader based on file type
if ext == ".pdf":
    # Travel guides, destination guides, tour itineraries, etc.
    loader = PyPDFLoader(file_path)

elif ext == ".txt":
    # Travel information, FAQs, destination descriptions, etc.
    loader = TextLoader(file_path)

elif ext == ".csv":
    # Tour packages, hotels, destinations, prices, attractions, etc.
    loader = CSVLoader(file_path)

elif ext == ".docx":
    # Travel plans, tourism reports, tour package documents, etc.
    loader = UnstructuredWordDocumentLoader(file_path)

else:
    raise ValueError(f"Unsupported Travel & Tourism file type: {ext}")

# Load Travel & Tourism documents
documents = loader.load()

print("Travel & Tourism Documents Loaded:", len(documents))

# Display sample tourism information
print("\nSample Travel & Tourism Content:")
print(documents[0].page_content[:1000])


Saving Travel_Tourism_RAG_Knowledge_Base (1).txt to Travel_Tourism_RAG_Knowledge_Base (1).txt
Uploaded Travel & Tourism File: Travel_Tourism_RAG_Knowledge_Base (1).txt
Travel & Tourism Documents Loaded: 1

Sample Travel & Tourism Content:
TRAVEL & TOURISM RAG KNOWLEDGE BASE
Version: 1.0
Purpose: General-purpose knowledge base for a Travel & Tourism RAG chatbot.

1. TRAVEL PLANNING
Travel planning involves selecting a destination, deciding travel dates, estimating a budget, arranging transport and accommodation, planning activities, and checking safety or entry requirements.
A useful itinerary normally contains destination, date, transport, accommodation, activities, meal breaks, estimated costs, and free time.
Travelers should verify prices, operating hours, weather, entry requirements, and local rules before a trip because these can change.

2. TYPES OF TOURISM
Leisure tourism: Travel mainly for relaxation, recreation, sightseeing, beaches, resorts, or entertainment.
Adventure tourism

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split Travel & Tourism documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# Create chunks from travel and tourism documents
travel_tourism_docs = text_splitter.split_documents(documents)

print("Total Travel & Tourism Chunks:", len(travel_tourism_docs))

# Display a sample chunk
print("\nSample Travel & Tourism Chunk:")
print(travel_tourism_docs[0].page_content[:500])


Total Travel & Tourism Chunks: 22

Sample Travel & Tourism Chunk:
TRAVEL & TOURISM RAG KNOWLEDGE BASE
Version: 1.0
Purpose: General-purpose knowledge base for a Travel & Tourism RAG chatbot.


In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

print("Total Chunks:", len(docs))

Total Chunks: 22


In [19]:
from google.colab import userdata
api_key=userdata.get('api_key')
client = openai.OpenAI(
    api_key=api_key,
    base_url="https://nexusapi.navigatelabs.ai"
)

In [29]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="rag"
)


In [30]:
for i, doc in enumerate(docs):

    text = doc.page_content

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    embedding = response.data[0].embedding

    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("Embeddings stored successfully!")

Embeddings stored successfully!


In [31]:
query = input("Ask your question: ")

Ask your question: which place is best to travel


In [32]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
)

query_embedding = query_response.data[0].embedding

In [33]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

In [34]:
retrieved_chunks = results["documents"][0]

for i, chunk in enumerate(retrieved_chunks):

    print(f"\nChunk {i+1}")
    print("-" * 50)

    print(chunk)


Chunk 1
--------------------------------------------------
11. TRAVELER FAQ
Q: How should I choose a destination?
A: Consider budget, travel dates, weather, interests, available time, transport, and entry requirements.

Q: How long should an itinerary be?
A: It depends on the destination and travel goals. A practical itinerary should allow realistic travel time and some flexibility.

Chunk 2
--------------------------------------------------
1. TRAVEL PLANNING
Travel planning involves selecting a destination, deciding travel dates, estimating a budget, arranging transport and accommodation, planning activities, and checking safety or entry requirements.
A useful itinerary normally contains destination, date, transport, accommodation, activities, meal breaks, estimated costs, and free time.
Travelers should verify prices, operating hours, weather, entry requirements, and local rules before a trip because these can change.

Chunk 3
--------------------------------------------------
6. A

In [35]:
context = "\n\n".join(retrieved_chunks)

In [36]:
prompt = f"""
Answer the question using the context below. If there is no context, you can answer on your own

Context:
{context}

Question:
{query}
"""

In [37]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

answer = response.choices[0].message.content

print("\nFINAL ANSWER")
print("=" * 50)

print(answer)


FINAL ANSWER
The best place to travel depends on your interests and preferences. For example:

- If you enjoy cultural heritage, historical monuments, and bustling markets, Delhi, Jaipur, and Agra are great options.
- For relaxing beaches, water activities, and nightlife, Goa is ideal.
- For nature, waterfalls, coffee plantations, and scenic landscapes, Chikkamagaluru or Gokarna are excellent choices.
- If you're interested in wildlife safaris and forest landscapes, Kabini is suitable.
- For Himalayan scenery, trekking, and spiritual experiences, Uttarakhand offers remarkable options.

Considering your interests, travel goals, and available time will help determine the best destination for you.


In [47]:
while True:

    query = input("\nAsk Question (type exit to quit): ")

    if query.lower() == "exit":
        break

    # Query embedding
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    query_embedding = query_response.data[0].embedding

    # Retrieve documents
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    retrieved_chunks = results["documents"][0]

    context = "\n\n".join(retrieved_chunks)

    # Prompt
    prompt = f"""
    Answer the question using the context below.

    Context:
    {context}

    Question:
    {query}
    """

    # LLM response
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response.choices[0].message.content

    print("\nAnswer:")
    print(answer)


Ask Question (type exit to quit): exit
